# 04 - Run all systems on all questions

One model (Qwen3.5-9B, 4-bit NF4, greedy decoding, thinking off), four systems, the 175 questions of
`questions/questions.jsonl` (ids Q001-Q175; `split` = dev / test / agentic, `type` = fact, numeric, comparison,
multi_hop, false_premise, unanswerable).

| system | retrieval | passages given to the model |
|---|---|---|
| `rag_base` | dense search over all filings | 5 |
| `rag5` | company/year filter + reranker | 5 |
| `rag8` | filter + BM25/dense hybrid + reranker | 8 |
| `agent_v3` | LangGraph agent: plan -> search -> checklist -> answer | 8 |

Set `VARIANT` in the first code cell to choose which chunks are used: `"text"` (tables as flat text, notebook 02a)
or `"tables"` (tables written value by value, notebook 02b). Run the notebook once per variant; the results go in
`runs/<variant>/`.

Every answered question is appended to `runs/<system>.jsonl` at once, and the folder is uploaded to the private
dataset (`runs/<variant>/`) every 20 questions and after each system. If Colab disconnects or the GPU limit is
reached, run the notebook again, also from **another Google account** (it needs its own `HF_TOKEN` secret): the files
are restored from the dataset first and finished questions are skipped (Runtime > Restart session, Run all).
The agent also records the seconds of each stage (`t_plan`, `t_search`, `t_check`, `t_answer`).

Before running: Runtime > T4 GPU, Colab Secret `HF_TOKEN` (notebook access on). Several hours in total; the agent is
the slowest. After the run the answers still have to be graded by hand (rule-based checks are only a first filter).

In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
VARIANT = "text"  # "text" or "tables"
assert VARIANT in {"text", "tables"}

In [ ]:
# Results are kept in the private dataset (runs/<variant>/), so another Colab session or another Google
# account can continue the run. Finished questions found there are not computed again.
from src.hub.runs import backup_runs, restore_runs

os.makedirs("runs", exist_ok=True)

In [ ]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import userdata
from sentence_transformers import SentenceTransformer

from src.agent.graph import AgentConfig, build_agent, regex_fallback
from src.agent.llm import generate, load_llm
from src.eval.golden import load_golden, validate
from src.eval.metrics import summarise
from src.eval.systems import (
    RAG5,
    RAG8,
    RAG_BASE,
    make_agent_system,
    make_rag_system,
    run_system,
)
from src.hub.dataset import download_folder
from src.retrieval.embed import embed_texts, load_chunks
from src.retrieval.pipeline import Corpus, scoped_search
from src.retrieval.rerank import Reranker
from src.retrieval.scope import filings_of

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/10k-agentic-rag-data"
print("restored from the dataset:", restore_runs(HF_REPO, VARIANT, Path("runs")))
download_folder(HF_REPO, "text", Path("data"))  # the questions are written for these chunks
if VARIANT == "tables":
    download_folder(HF_REPO, "tables", Path("data"))
folder = Path("data") / VARIANT
meta = json.loads((folder / "meta.json").read_text())
chunk_file = "chunks_1600.jsonl" if VARIANT == "text" else "chunks_tables_1600.jsonl"
chunks = load_chunks(folder / chunk_file)
embeddings = np.load(folder / "embeddings_bge-m3.npy").astype(np.float32)
assert len(chunks) == len(embeddings) == meta["n"]

questions = load_golden("questions/questions.jsonl")
text_chunks = load_chunks("data/text/chunks_1600.jsonl")
errors, _ = validate(questions, text_chunks)
assert not errors, errors
if VARIANT == "tables":  # the gold evidence is written for the text chunks: translate it by content
    from src.eval.remap import remap_questions

    questions, weak = remap_questions(
        questions, text_chunks, chunks, min_cover=0.7, focus_on_answer=True
    )
    print("evidence matched by content; weak matches:", weak)
print(
    VARIANT,
    len(questions),
    "questions:",
    pd.Series([q["split"] for q in questions]).value_counts().to_dict(),
)

corpus = Corpus(chunks, embeddings)
filings = filings_of(chunks)
names = {c["ticker"]: c["company"] for c in chunks}
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
embedder.max_seq_length = meta["max_seq_length"]
reranker = Reranker(max_length=meta["max_seq_length"])

In [ ]:
MODEL = "Qwen/Qwen3.5-9B"
gc.collect()
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb > 9, "GPU memory is taken by an earlier run: Runtime > Restart session, Run all"
tokenizer, model, loader = load_llm(MODEL)
print("loaded with", loader)


def llm_text(messages, max_new_tokens):
    return generate(tokenizer, model, messages, max_new_tokens)["text"]


def llm_full(messages):
    return generate(tokenizer, model, messages)


def embed_query(text):
    return embed_texts(embedder, [text], progress=False)[0]


def search(query, ticker, year, k):
    return scoped_search(corpus, embed_query(query), query, ticker, year, k, reranker)


def score_fn(question, passages):
    return reranker.score(question, [c["embed_text"] for c in passages])


agent = build_agent(
    llm_text,
    search,
    filings,
    names,
    AgentConfig(),
    default_scope=regex_fallback(filings),
    score_fn=score_fn,
)
rag = {
    "corpus": corpus,
    "embed_query": embed_query,
    "reranker": reranker,
    "generate": llm_full,
    "model_name": MODEL,
}
SYSTEMS = {
    "rag_base": make_rag_system("rag_base", RAG_BASE, 5, **rag),
    "rag5": make_rag_system("rag5", RAG5, 5, **rag),
    "rag8": make_rag_system("rag8", RAG8, 8, **rag),
    "agent_v3": make_agent_system("agent_v3", agent, model_name=MODEL),
}

Quick check on three questions before the long run (a broken setup shows up here, not after hours).
The files are removed afterwards, so the real run starts clean.

In [ ]:
import shutil

for name, fn in SYSTEMS.items():
    rows = run_system(questions[:3], fn, f"runs_smoke/{name}.jsonl")
    print(name, [(r["id"], r["seconds"]) for r in rows])
shutil.rmtree("runs_smoke")

Full run: all 175 questions, one system after the other (cheapest first, so a GPU limit costs the least).

In [ ]:
BACKUP_EVERY = 20  # questions; the agent needs about 40 s per question
results = {}
for name, fn in SYSTEMS.items():
    counter = {"n": 0}

    def log(message, counter=counter):
        print(message)
        counter["n"] += 1
        if counter["n"] % BACKUP_EVERY == 0:
            backup_runs(HF_REPO, VARIANT, Path("runs"))

    results[name] = run_system(questions, fn, f"runs/{name}.jsonl", log=log)
    backup_runs(HF_REPO, VARIANT, Path("runs"))

In [ ]:
pd.set_option("display.width", 220)
columns = [
    "n",
    "behaviour_ok",
    "refused",
    "cites_evidence",
    "numbers_match",
    "hit@5",
    "all@5",
    "all@10",
    "all@k_given",
    "prompt_tokens",
    "seconds",
]
summaries = {}
for split in ["dev", "test", "agentic"]:
    for name, rows in results.items():
        part = [r for r in rows if r["split"] == split]
        summaries[f"{split}_{name}"] = summarise(part)
        print(f"\n{split} / {name}")
        print(pd.DataFrame(summaries[f"{split}_{name}"]).T.reindex(columns=columns).to_string())
os.makedirs("runs", exist_ok=True)
with open("runs/summary.json", "w") as f:
    json.dump(summaries, f, indent=1)

Where does the agent's time go? Mean seconds per question for each stage (answer = the final LLM call).

In [ ]:
agent_rows = pd.DataFrame(results["agent_v3"])
stages = ["t_plan", "t_search", "t_check", "t_answer", "seconds"]
print(agent_rows[stages].describe().loc[["mean", "50%", "max"]].round(1).to_string())
print(agent_rows.groupby("split")[stages].mean().round(1).to_string())

Faster agent (optional, after the full run): `agent_v3_fast` skips the check step when the plan has a single
search (one company, one year). Only these questions change, so only these are run again (about 90 of 175; the other
questions behave exactly as in `agent_v3`). Compared on the same questions: seconds, answers that stay identical,
and the rule-based checks. Whether the answers are still correct is decided by hand grading, as for the other systems.


In [ ]:
from src.eval.systems import single_plan_ids

agent_fast = build_agent(
    llm_text,
    search,
    filings,
    names,
    AgentConfig(skip_check_single=True),
    default_scope=regex_fallback(filings),
    score_fn=score_fn,
)
with open("runs/agent_v3.jsonl") as f:
    base_rows = [json.loads(line) for line in f if line.strip()]
affected = set(single_plan_ids(base_rows))
subset = [q for q in questions if q["id"] in affected]
fast_rows = run_system(
    subset,
    make_agent_system("agent_v3_fast", agent_fast, model_name=MODEL),
    "runs/agent_v3_fast.jsonl",
)

backup_runs(HF_REPO, VARIANT, Path("runs"))

base = {r["id"]: r for r in base_rows}
same = sum(r["answer"] == base[r["id"]]["answer"] for r in fast_rows)
print(len(fast_rows), "questions;", same, "with an identical answer")
for label, rows in [("agent_v3", [base[r["id"]] for r in fast_rows]), ("agent_v3_fast", fast_rows)]:
    print(
        label,
        "mean seconds",
        round(sum(r["seconds"] for r in rows) / len(rows), 1),
        "| behaviour_ok",
        round(sum(r["behaviour_ok"] for r in rows) / len(rows), 3),
    )

In [ ]:
import shutil

from google.colab import files

shutil.make_archive(f"runs_{VARIANT}", "zip", "runs")
files.download(f"runs_{VARIANT}.zip")

Next: hand-grade the answers (blind, correct / partial / wrong), then compute the final tables.